# BrainDecoding — Google Colab GPU 留出测试集 (`test`) 评估、负对照与可视化工作流

本 Notebook 专门用于在 **Google Colab GPU 实例** 上对已完成训练的模型执行**留出测试集 (`split=test`) 全面评估、时间错位/全零信号负对照检验与可视化对比**：
- **支持三阶段完整横向对比（`--all-stages`）**：
  - **阶段 1 (`word`)**：词级脑编码器（纯 CNN，`use_transformer: false`）
  - **阶段 2.1 (`context`)**：词级脑编码器 + 16 层全局上下文 Transformer（`use_transformer: true`，完整窗口，无局部滚动修订）
  - **阶段 2.2 (`revision`)**：词级脑编码器 + 16 层局部窗口滚动修订（`use_transformer: true`，`K0/K1/K2` 局部窗口滚动回修）
- **支持独立解耦的负对照配置体系（`configs/controls/`）**：
  - **时间错位真实信号负对照（Temporal Shift Control）**：在同受试者、同连续录音内对真实脑磁/脑电信号施加 $\pm 30\text{ s}$、$\pm 60\text{ s}$、$\pm 120\text{ s}$（`[-120, -60, -30, +30, +60, +120]` 秒）确定性圆周移位，配置文件位于 `configs/controls/temporal_shift/`；
  - **全零神经信号负对照（Zero-Signal Control）**：将输入脑信号置零以检验无脑信号时的机遇基线，配置文件位于 `configs/controls/zero_signal/`；
- **支持 `EXP_TAG` 与 `WORD_EXP_TAG` 跨实验标签组合评估**：按 `runs/<dataset>/<EXP_TAG>/<stage>/...` 自动读取对应超参实验的权重，且支持通过 `WORD_EXP_TAG` 引用不同实验标签下的阶段 1 (`word`) 参与三阶段对比；
- **自动输出多维度定量报表与高清可视化图表**：
  1. 多词表规模检索曲线（`N=20, 50, 100, 150, Full Vocab` 的 Macro/Micro Top-10、Top-1、MRR、中位秩 vs 随机基线）；
  2. 真实对齐脑信号（`Clean 0s`）对比 $\pm 30\text{ s}, \pm 60\text{ s}, \pm 120\text{ s}$ 错位脑信号负对照的检索准确率与降幅对比图（`negative_control_test.png` 与 `stage_control_comparison_test.png`）；
  3. 分受试者（`sub-01` .. `sub-N`）测试集 50 词检索准确率拆解；
  4. 阶段 2.2 滚动修订各延迟视野（`K0 -> K1 -> K2`）测试集增益拆解。

In [ ]:
# 1. 检查 GPU 并挂载 Google Drive、配置环境变量
!nvidia-smi

import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

# ==================== 用户可修改参数 ====================
GIT_REPO_URL = 'https://github.com/wdongdong0708-jpg/BrainDecoding2.git'
GIT_BRANCH = 'master'
DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'

# 选择要评估的数据集或单阶段 YAML 配置文件：
#   - 'configs/training/pallier2025_word_encoder.yaml'          (法语 MEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/pallier2025_context_transformer.yaml'   (法语 MEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/pallier2025_rolling_revision.yaml'      (法语 MEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
#   - 'configs/training/chineseeeg2_word_encoder.yaml'          (中文 EEG 阶段 1:   词级脑编码器 CNN)
#   - 'configs/training/chineseeeg2_context_transformer.yaml'   (中文 EEG 阶段 2.1: 词级脑编码器 + 16层全局上下文 Transformer)
#   - 'configs/training/chineseeeg2_rolling_revision.yaml'      (中文 EEG 阶段 2.2: 词级脑编码器 + 16层局部窗口滚动修订)
DATASET = 'pallier2025'               # 'pallier2025' 或 'chineseeeg2_littleprince'
CONFIG_FILE = 'configs/training/pallier2025_rolling_revision.yaml'
MODALITY = 'meg'                      # 'meg' (pallier2025) 或 'eeg' (chineseeeg2)

# 选择独立解耦的负对照配置文件（位于 configs/controls/，与 configs/word_decoding/ 解耦）：
#   - 'configs/controls/temporal_shift/temporal_shift_control.yaml'             (通用 +-30s, +-60s, +-120s 真实脑信号时间错位负对照)
#   - 'configs/controls/temporal_shift/pallier2025_temporal_shift_control.yaml'  (Pallier2025 专属 +-30s, +-60s, +-120s 时间错位负对照)
#   - 'configs/controls/temporal_shift/chineseeeg2_temporal_shift_control.yaml'  (ChineseEEG-2 专属 +-30s, +-60s, +-120s 时间错位负对照)
#   - 'configs/controls/zero_signal/zero_signal_control.yaml'                   (全零脑磁/脑电信号负对照)
#   - ''                                                                        (留空 '' 则仅运行标准 Clean 0s 评估)
CONTROL_CONFIG = 'configs/controls/temporal_shift/temporal_shift_control.yaml'

EXP_TAG = 'lr1e-4_bs128'              # 待评估的超参实验标签（定位 runs/<dataset>/<EXP_TAG>/<stage>/...；留空 '' 则为 runs/<dataset>/<stage>/...）
WORD_EXP_TAG = ''                     # 若阶段 1 (word) 保存在另一个实验标签下，在此填入该标签名（留空 '' 默认与 EXP_TAG 相同；填 'default' 定位无标签根目录 word）
SEED = 0                              # 随机种子: 0, 1, 2
SUBSET_ARGS = '--subject all'         # 必须与训练时的受试者子集保持一致
# ========================================================

CONTROL_ARGS = f'--control-config {CONTROL_CONFIG}' if str(CONTROL_CONFIG).strip() else ''

CODE_DIR = '/content/code'
VENV_DIR = '/content/proj-venv'
PYTHON = f'{VENV_DIR}/bin/python'
LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'

DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

for folder in (
    LINKS_DIR,
    LOCAL_CACHE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDATA_ROOT'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR
os.environ['BRAINDECODING_EXP_TAG'] = EXP_TAG
os.environ['BRAINDECODING_WORD_EXP_TAG'] = WORD_EXP_TAG
os.environ['BRAINDECODING_MODEL_DIR'] = DRIVE_MODELS_DIR

print('DATASET                   =', DATASET)
print('CONFIG_FILE               =', CONFIG_FILE)
print('CONTROL_CONFIG            =', CONTROL_CONFIG or '(disabled - clean only)')
print('BRAINDECODING_RESULTS_DIR =', os.environ['BRAINDECODING_RESULTS_DIR'])
print('BRAINDECODING_EXP_TAG     =', os.environ['BRAINDECODING_EXP_TAG'] or '(default)')
print('BRAINDECODING_WORD_EXP_TAG=', os.environ['BRAINDECODING_WORD_EXP_TAG'] or '(same as EXP_TAG / fallback default)')

In [ ]:
# 2. 克隆或同步最新代码到 /content/code
!if [ ! -d "{CODE_DIR}/.git" ]; then \
    git clone -b {GIT_BRANCH} {GIT_REPO_URL} {CODE_DIR}; \
else \
    git -C {CODE_DIR} fetch origin && git -C {CODE_DIR} checkout {GIT_BRANCH} && git -C {CODE_DIR} pull --ff-only; \
fi

In [ ]:
# 3. 初始化虚拟环境 + 建立 /content/studies_links 软链接桥
!python3 -u {CODE_DIR}/scripts/colab_setup.py \
    --venv-dir {VENV_DIR} \
    --project-root {CODE_DIR} \
    --drive-data-dir {DRIVE_DATA_DIR} \
    --links-dir {LINKS_DIR} \
    --local-cache-dir {LOCAL_CACHE_DIR}

In [ ]:
# 4. 秒级解包 Google Drive 上的 .tar 缓存归档回 /content/local_cache
!cd {CODE_DIR} && {PYTHON} -u scripts/cache_archive.py unpack \
    --archive-dir {DRIVE_ARCHIVE_DIR} \
    --cache-dir {LOCAL_CACHE_DIR} \
    --modality {MODALITY} \
    --skip-existing

In [ ]:
# 5. 【推荐】一键评估当前数据集与 EXP_TAG 下所有已训练阶段（阶段 1 word / 阶段 2.1 context / 阶段 2.2 revision）
#    当设置了 CONTROL_CONFIG 时，自动对每个阶段同时运行 Clean (0s) 与 +-30s, +-60s, +-120s 错位信号负对照并输出对比图
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
    --dataset {DATASET} \
    --seed {SEED} \
    --split test \
    --all-stages \
    {CONTROL_ARGS} \
    {SUBSET_ARGS}

from IPython.display import Image, display

test_pngs = sorted(
    Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(SEED):03d}/figures/*test*.png'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
allowed_tags = {t for t in (EXP_TAG, WORD_EXP_TAG) if t and t.lower() not in ('default', 'root', 'none')}
shown = set()
for png_path in test_pngs:
    if allowed_tags and not any(t in png_path.parts for t in allowed_tags):
        continue
    key = (png_path.parent.parent.parent.name, png_path.name)
    if key in shown:
        continue
    shown.add(key)
    print(f'\n[测试集图表] {png_path}')
    display(Image(filename=str(png_path)))

In [ ]:
# 6. 【可选】仅评估单个指定的 CONFIG_FILE 在测试集 (test) 上的表现、负对照与可视化看板
!cd {CODE_DIR} && {PYTHON} -u -m braindecoding eval-run \
    --config {CONFIG_FILE} \
    --seed {SEED} \
    --split test \
    {CONTROL_ARGS} \
    {SUBSET_ARGS}

from IPython.display import Image, display

single_pngs = [
    p for p in sorted(
        Path(DRIVE_RESULTS_DIR).rglob(f'seed-{int(SEED):03d}/figures/*test*.png'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not EXP_TAG or EXP_TAG in p.parts
][:3]
for png_path in single_pngs:
    print(f'\n[单阶段测试集图表] {png_path}')
    display(Image(filename=str(png_path)))